### ============================================
### NOTEBOOK : 03_data_split.ipynb
### ============================================

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path


In [2]:
#chargement du dataset nettoyé
data_merged=pd.read_csv('../data/staging/data_merged.csv')
print(f"Données chargées:{len(data_merged):,}lignes")

Données chargées:83,692lignes


### ============================
### 2. Split par persona
### ============================

In [3]:
print("/!\! SPLIT PAR PERSONA /!\!")
# Split 70% train 15% val 15% test
train_data, temp_data=train_test_split(data_merged,test_size=0.3,stratify=data_merged['persona_target'],random_state=42) 
val_data,test_data=train_test_split(temp_data,test_size=0.5,stratify=temp_data['persona_target'],random_state=42)

/!\! SPLIT PAR PERSONA /!\!


In [6]:
print(f"\n Split terminé :")
print(f"  - Train : {len(train_data):,} lignes ({len(train_data)/len(data_merged)*100:.1f}%)")
print(f"  - Val   : {len(val_data):,} lignes ({len(val_data)/len(data_merged)*100:.1f}%)")
print(f"  - Test  : {len(test_data):,} lignes ({len(test_data)/len(data_merged)*100:.1f}%)")


 Split terminé :
  - Train : 58,584 lignes (70.0%)
  - Val   : 12,554 lignes (15.0%)
  - Test  : 12,554 lignes (15.0%)


In [4]:
# Vérification de la répartition des personas dans chaque split
print("\nRépartition des personas :")
print("train:")
print(train_data['persona_target'].value_counts())
print("Val:")
print(val_data['persona_target'].value_counts())
print("Test:")
print(test_data['persona_target'].value_counts())


Répartition des personas :
train:
persona_target
Infirmier      32618
Radiologue     12662
Autre           5698
Urgentiste      5692
Généraliste     1914
Name: count, dtype: int64
Val:
persona_target
Infirmier      6990
Radiologue     2713
Autre          1221
Urgentiste     1220
Généraliste     410
Name: count, dtype: int64
Test:
persona_target
Infirmier      6990
Radiologue     2713
Autre          1221
Urgentiste     1220
Généraliste     410
Name: count, dtype: int64


### --------------------------------------------
### 4. SAUVEGARDER
### --------------------------------------------

In [5]:
train_data.to_csv('../data/processed/train_data.csv',index=False)
val_data.to_csv('../data/processed/val_data.csv',index=False)
test_data.to_csv('../data/processed/test_data.csv',index=False)
print("data sauvegardées dans ../data/processed/")

data sauvegardées dans ../data/processed/


### ============================================
### SPLIT data_merged.csv EN TRAIN/VAL/TEST
### ============================================

In [ ]:


import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 60)
print(" SPLIT ROBUSTE (gestion automatique petites classes)")
print("=" * 60)

# --------------------------------------------
# 1. CHARGER data_merged.csv
# --------------------------------------------

print("\n Chargement data_merged.csv...")

data_merged = pd.read_csv("../data/staging/data_merged.csv")

print(f" Données chargées : {len(data_merged):,} lignes")

# --------------------------------------------
# 2. HOSPITALISATIONS UNIQUES
# --------------------------------------------

print("\n Analyse des hospitalisations...")

# Une hospitalisation = un HADM_ID (prendre le persona le plus fréquent)
hospitalisations = data_merged.groupby('HADM_ID').agg({
    'persona_target': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'SUBJECT_ID': 'first'
}).reset_index()

print(f" Hospitalisations uniques : {len(hospitalisations):,}")

# Distribution
persona_counts = hospitalisations['persona_target'].value_counts()
print(f"\n Distribution par persona (hospitalisations) :")
for persona, count in persona_counts.items():
    print(f"  {persona:15s} : {count:4,} hospitalisations")

# --------------------------------------------
# 3. VÉRIFIER MINIMUM PAR PERSONA
# --------------------------------------------

print("\n Vérification des minimums pour split...")

MIN_PER_SPLIT = 2  # Minimum 2 par train/val/test

# Calculer combien on aura dans val et test (15% chacun)
min_needed = {}
for persona, count in persona_counts.items():
    val_count = int(count * 0.15)
    test_count = int(count * 0.15)
    min_needed[persona] = (val_count, test_count)
    
    status = "" if val_count >= MIN_PER_SPLIT and test_count >= MIN_PER_SPLIT else ""
    print(f"  {status} {persona:15s} : Val={val_count}, Test={test_count}")

# Identifier personas problématiques
problematic = [p for p, (v, t) in min_needed.items() 
               if v < MIN_PER_SPLIT or t < MIN_PER_SPLIT]

if problematic:
    print(f"\n Personas avec trop peu d'hospitalisations : {problematic}")
    print(f"   → Vont être fusionnés avec 'Autre'")
    
    # Fusionner dans Autre
    hospitalisations.loc[hospitalisations['persona_target'].isin(problematic), 'persona_target'] = 'Autre'
    data_merged.loc[data_merged['persona_target'].isin(problematic), 'persona_target'] = 'Autre'
    
    # Recalculer
    persona_counts = hospitalisations['persona_target'].value_counts()
    print(f"\n Distribution APRÈS fusion :")
    for persona, count in persona_counts.items():
        print(f"  {persona:15s} : {count:4,} hospitalisations")

# --------------------------------------------
# 4. SPLIT MANUEL PAR PERSONA
# --------------------------------------------

print("\n Split manuel par persona...")

train_hadm_list = []
val_hadm_list = []
test_hadm_list = []

for persona in persona_counts.index:
    # Filtrer hospitalisations de ce persona
    persona_hadm = hospitalisations[hospitalisations['persona_target'] == persona].copy()
    
    # Mélanger
    persona_hadm = persona_hadm.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Calculer splits
    n = len(persona_hadm)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    n_test = n - n_train - n_val  # Le reste pour éviter erreurs d'arrondi
    
    # Split
    train_p = persona_hadm.iloc[:n_train]
    val_p = persona_hadm.iloc[n_train:n_train+n_val]
    test_p = persona_hadm.iloc[n_train+n_val:]
    
    print(f"  {persona:15s} : Train={len(train_p):4,} | Val={len(val_p):4,} | Test={len(test_p):4,}")
    
    # Ajouter aux listes
    train_hadm_list.append(train_p)
    val_hadm_list.append(val_p)
    test_hadm_list.append(test_p)

# Concaténer
train_hadm = pd.concat(train_hadm_list, ignore_index=True)
val_hadm = pd.concat(val_hadm_list, ignore_index=True)
test_hadm = pd.concat(test_hadm_list, ignore_index=True)

print(f"\n Split terminé :")
print(f"  - Train : {len(train_hadm):,} hospitalisations ({len(train_hadm)/len(hospitalisations)*100:.1f}%)")
print(f"  - Val   : {len(val_hadm):,} hospitalisations ({len(val_hadm)/len(hospitalisations)*100:.1f}%)")
print(f"  - Test  : {len(test_hadm):,} hospitalisations ({len(test_hadm)/len(hospitalisations)*100:.1f}%)")

# --------------------------------------------
# 5. RÉCUPÉRER TOUTES LES NOTES
# --------------------------------------------

print("\n🔗 Récupération des notes...")

train_data = data_merged[data_merged['HADM_ID'].isin(train_hadm['HADM_ID'])].copy()
val_data = data_merged[data_merged['HADM_ID'].isin(val_hadm['HADM_ID'])].copy()
test_data = data_merged[data_merged['HADM_ID'].isin(test_hadm['HADM_ID'])].copy()

print(f"\n Notes récupérées :")
print(f"  - Train : {len(train_data):,} notes ({len(train_data)/len(data_merged)*100:.1f}%)")
print(f"  - Val   : {len(val_data):,} notes ({len(val_data)/len(data_merged)*100:.1f}%)")
print(f"  - Test  : {len(test_data):,} notes ({len(test_data)/len(data_merged)*100:.1f}%)")

# --------------------------------------------
# 6. VÉRIFICATIONS FINALES
# --------------------------------------------

print("\n VÉRIFICATIONS FINALES :")

# Doublons HADM_ID
train_hadm_set = set(train_data['HADM_ID'])
val_hadm_set = set(val_data['HADM_ID'])
test_hadm_set = set(test_data['HADM_ID'])

overlap_train_val = train_hadm_set & val_hadm_set
overlap_train_test = train_hadm_set & test_hadm_set
overlap_val_test = val_hadm_set & test_hadm_set

print(f"\n Doublons HADM_ID :")
print(f"  - Train ∩ Val : {len(overlap_train_val)} {'' if len(overlap_train_val)==0 else ''}")
print(f"  - Train ∩ Test : {len(overlap_train_test)} {'' if len(overlap_train_test)==0 else ''}")
print(f"  - Val ∩ Test : {len(overlap_val_test)} {'' if len(overlap_val_test)==0 else '❌'}")

# Distribution
print(f"\n Distribution finale par persona :")
print("\n TRAIN :")
print(train_data['persona_target'].value_counts())
print("\n VAL :")
print(val_data['persona_target'].value_counts())
print("\n TEST :")
print(test_data['persona_target'].value_counts())

# Valeurs manquantes
print(f"\n Valeurs manquantes :")
print(f"  - Train : {train_data[['TEXT_CLEAN', 'diagnosis_labels', 'persona_target']].isnull().sum().sum()}")
print(f"  - Val   : {val_data[['TEXT_CLEAN', 'diagnosis_labels', 'persona_target']].isnull().sum().sum()}")
print(f"  - Test  : {test_data[['TEXT_CLEAN', 'diagnosis_labels', 'persona_target']].isnull().sum().sum()}")

# --------------------------------------------
# 7. SAUVEGARDER
# --------------------------------------------

print("\n Sauvegarde des fichiers...")

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

train_data.to_csv(processed_path / 'train_data.csv', index=False)
val_data.to_csv(processed_path / 'val_data.csv', index=False)
test_data.to_csv(processed_path / 'test_data.csv', index=False)

print(f"\n Fichiers sauvegardés dans {processed_path}/ :")
print(f"  - train_data.csv ({len(train_data):,} lignes)")
print(f"  - val_data.csv ({len(val_data):,} lignes)")
print(f"  - test_data.csv ({len(test_data):,} lignes)")

# Tailles
train_size = (processed_path / 'train_data.csv').stat().st_size / 1024 / 1024
val_size = (processed_path / 'val_data.csv').stat().st_size / 1024 / 1024
test_size = (processed_path / 'test_data.csv').stat().st_size / 1024 / 1024

print(f"\n Tailles fichiers :")
print(f"  - train_data.csv : {train_size:.1f} MB")
print(f"  - val_data.csv : {val_size:.1f} MB")
print(f"  - test_data.csv : {test_size:.1f} MB")
print(f"  - TOTAL : {train_size + val_size + test_size:.1f} MB")

print("\n" + "=" * 60)
print(" SPLIT RÉUSSI !")
print("=" * 60)

print(f"\n Split propre sans doublons")
print(f" Distribution équilibrée par persona")


🔧 SPLIT ROBUSTE (gestion automatique petites classes)

📂 Chargement data_merged.csv...
✅ Données chargées : 83,692 lignes

🔍 Analyse des hospitalisations...
✅ Hospitalisations uniques : 2,776

📊 Distribution par persona (hospitalisations) :
  Infirmier       : 1,482 hospitalisations
  Radiologue      :  793 hospitalisations
  Urgentiste      :  282 hospitalisations
  Autre           :  125 hospitalisations
  Généraliste     :   94 hospitalisations

🔍 Vérification des minimums pour split...
  ✅ Infirmier       : Val=222, Test=222
  ✅ Radiologue      : Val=118, Test=118
  ✅ Urgentiste      : Val=42, Test=42
  ✅ Autre           : Val=18, Test=18
  ✅ Généraliste     : Val=14, Test=14

🔀 Split manuel par persona...
  Infirmier       : Train=1,037 | Val= 222 | Test= 223
  Radiologue      : Train= 555 | Val= 118 | Test= 120
  Urgentiste      : Train= 197 | Val=  42 | Test=  43
  Autre           : Train=  87 | Val=  18 | Test=  20
  Généraliste     : Train=  65 | Val=  14 | Test=  15

✅ Split 